# Boundary data: lifting, mixed conditions, and the Neumann kernel

These examples implement the weak formulations from Chapter 4 on the same unit square $\Omega=(0,1)^2$.

1. **Nonhomogeneous Dirichlet data:** construct a lifting and solve for a homogeneous correction.
2. **Mixed boundary conditions:** combine zero Dirichlet data with nonzero Neumann and Robin data.
3. **The pure Neumann problem:** identify the constant kernel, check compatibility, and impose mean zero.
4. **Convection:** compare a discrete loss of stability with large divergence-free flow and boundary layers.

Throughout, $n$ is the **outward** unit normal and the Neumann datum follows the convention

$$
\begin{align*}
h_N=n\cdot(a\nabla u).
\end{align*}
$$

The physical diffusive flux would be $q=-a\nabla u$, so $q\cdot n=-h_N$.

Run the cells in order. We start with linear finite elements. Set `order = 2` below and rerun the notebook to check that the quadratic exact solution in Part 2 is reproduced up to rounding error. The exact solution in Part 1 has degree four.

**Packages:** NGSolve/Netgen and its WebGUI, NumPy, and Matplotlib.

In [ ]:
from ngsolve import *
from netgen.occ import unit_square
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

order = 1
maxh = 0.1
mesh = Mesh(unit_square.GenerateMesh(maxh=maxh))

print("Boundary names:", mesh.GetBoundaries())
print("Finite element order:", order)
Draw(mesh);

We report the $L^2$ error and the **full** $H^1$ error

$$
\begin{align*}
\|e\|_{H^1(\Omega)}^2=\int_\Omega |e|^2\,dx+\int_\Omega |\nabla e|^2\,dx.
\end{align*}
$$

In [ ]:
def error_norms(uh, exact):
    grad_exact = CF((exact.Diff(x), exact.Diff(y)))
    l2_squared = Integrate((uh - exact)**2, mesh)
    gradient_squared = Integrate((grad(uh) - grad_exact)**2, mesh)
    return sqrt(l2_squared), sqrt(l2_squared + gradient_squared)

## 1. Nonhomogeneous Dirichlet data and lifting

We solve

$$
\begin{align*}
-\Delta u=f\quad\text{in }\Omega,\qquad u=g\quad\text{on }\partial\Omega.
\end{align*}
$$

Choose

$$
\begin{align*}
u_{\rm ex}&=1+x+y+x(1-x)y(1-y),\\
f&=2\bigl(x(1-x)+y(1-y)\bigr),\qquad g=1+x+y.
\end{align*}
$$

Note that the bubble $x(1-x)y(1-y)$ vanishes on the whole boundary. It therefore changes the interior solution without changing the Dirichlet data.

In [ ]:
bubble = CF(x*(1-x)*y*(1-y))
u_D_exact = CF(1 + x + y + bubble)
g_D = CF(1 + x + y)
f_D = -(u_D_exact.Diff(x).Diff(x) + u_D_exact.Diff(y).Diff(y))
Draw(u_D_exact, mesh, "exact Dirichlet solution");

The trial functions have trace $g$, whereas the test functions have zero trace. Choose a discrete lifting $w_{g,h}$ with the prescribed boundary coefficients and write

$$
\begin{align*}
u_h=w_{g,h}+u_{0,h},\qquad u_{0,h}\in V_{h,0}.
\end{align*}
$$

The correction solves
$$
\begin{align*}
\int_\Omega\nabla u_{0,h}\cdot\nabla v_h\,dx=\int_\Omega f v_h\,dx-\int_\Omega\nabla w_{g,h}\cdot\nabla v_h\,dx\quad\forall v_h\in V_{h,0}.
\end{align*}
$$

The Dirichlet flags specify which coefficients are constrained. Assembly still creates matrices and vectors for **all** coefficients. `FreeDofs()` selects the unknown correction of the free degrees of freedom coefficients when solving.

In [ ]:
V_D = H1(mesh, order=order, dirichlet=".*")
u_D, v_D = V_D.TnT()

a_D = BilinearForm(V_D, symmetric=True)
a_D += grad(u_D)*grad(v_D)*dx
l_D = LinearForm(V_D)
l_D += f_D*v_D*dx

a_D.Assemble()
l_D.Assemble()
inverse_D = a_D.mat.Inverse(V_D.FreeDofs(), inverse="sparsecholesky")

A new `GridFunction` starts with zero coefficients. `Set(g_D, BND)` sets an extension of the prescribed boundary values. This extension is generally **not** the minimum-norm lifting, but it is stable.

In [ ]:
w_D = GridFunction(V_D, name="boundary lifting")
w_D.Set(g_D, BND)

boundary_error_D = sqrt(Integrate((w_D-g_D)**2, mesh, definedon=mesh.Boundaries(".*")))
print(f"Boundary L2 error of the lifting: {boundary_error_D:.3e}")
Draw(w_D, mesh, "one lifting of the boundary data");

Let $F$ denote the free degrees of freedom and $D$ the constrained ones. The coefficient vector of the correction satisfies

$$
\begin{align*}
A_{FF}\mathbf u_{0,F}=(\mathbf b-A\mathbf w_g)_F,\qquad \mathbf u_{0,D}=0.
\end{align*}
$$

Thus we subtract the contribution of the lifting **before** applying the inverse on the free degrees of freedom, and then add the lifting back.

In [ ]:
rhs_D = l_D.vec.CreateVector()
rhs_D.data = l_D.vec - a_D.mat*w_D.vec

u0_D = GridFunction(V_D, name="homogeneous correction")
u0_D.vec.data = inverse_D*rhs_D

uh_D = GridFunction(V_D, name="Dirichlet solution")
uh_D.vec.data = w_D.vec + u0_D.vec

l2_D, h1_D = error_norms(uh_D, u_D_exact)
print(f"L2 error: {l2_D:.3e}; full H1 error: {h1_D:.3e}")
Draw(u0_D, mesh, "homogeneous correction");
Draw(uh_D, mesh, "lifting plus correction");

### Does the choice of lifting matter?

Add any $z_h\in V_{h,0}$ to the first lifting. The correction changes, but the final solution stays the same

$$
\begin{align*}
w_{g,h}\longmapsto w_{g,h}+z_h,\qquad u_{0,h}\longmapsto u_{0,h}-z_h.
\end{align*}
$$

Below we project the bubble into the finite element space and explicitly zero its constrained coefficients. This step ensures that the **discrete** perturbation has zero trace: a volume projection of a zero-trace smooth function does not automatically have zero discrete boundary coefficients.

In [ ]:
z_D = GridFunction(V_D)
# preserve zero on the Dirichlet boundary
z_D.Set(20*bubble, dual=True)
Draw(z_D, mesh)

w_D_second = GridFunction(V_D, name="second lifting")
w_D_second.vec.data = w_D.vec + z_D.vec

rhs_D_second = l_D.vec.CreateVector()
rhs_D_second.data = l_D.vec - a_D.mat*w_D_second.vec
uh_D_second = GridFunction(V_D)
uh_D_second.vec.data = w_D_second.vec + inverse_D*rhs_D_second

lifting_trace_difference = sqrt(Integrate((w_D_second-w_D)**2, mesh, definedon=mesh.Boundaries(".*")))
lifting_difference = sqrt(Integrate((w_D_second-w_D)**2, mesh))
lifting_same_solution_h1 = sqrt(Integrate((uh_D_second-uh_D)**2 + (grad(uh_D_second)-grad(uh_D))**2, mesh))
print(f"Boundary difference between the liftings: {lifting_trace_difference:.3e}")
print(f"Volume difference between the liftings: {lifting_difference:.3e}")
print(f"H1 difference between the final solutions: {lifting_same_solution_h1:.3e}")
Draw(w_D_second, mesh, "a different lifting of the same data");

### The minimum full-H1-norm lifting

The quotient norm uses the **full** $H^1$ norm. Its discrete analogue minimizes that norm among all functions with the prescribed discrete trace. The minimizer satisfies

$$
\begin{align*}
\int_\Omega\bigl(\nabla u_{g,h}\cdot\nabla v_h+u_{g,h}v_h\bigr)\,dx=0
\quad\forall v_h\in V_{h,0}.
\end{align*}
$$

This is a separate problem with the stiffness-plus-mass matrix, corresponding to $-\Delta u_g+u_g=0$. We can compute it by correcting the same initial boundary extension. Its norm should be no larger than the norms of our other discrete liftings.

In [ ]:
u_H1, v_H1 = V_D.TnT()
a_H1 = BilinearForm(V_D, symmetric=True)
a_H1 += (grad(u_H1)*grad(v_H1) + u_H1*v_H1)*dx
a_H1.Assemble()

u_g_min = GridFunction(V_D, name="minimum H1 norm lifting")
u_g_min.vec.data = w_D.vec
rhs_H1 = u_g_min.vec.CreateVector()
rhs_H1.data = a_H1.mat*u_g_min.vec
u_g_min.vec.data -= a_H1.mat.Inverse(V_D.FreeDofs(), inverse="sparsecholesky")*rhs_H1

lifting_norms = []
for name, lifting in [("first lifting", w_D),
                      ("second lifting", w_D_second),
                      ("minimum H1 lifting", u_g_min)]:
    value = sqrt(Integrate(lifting**2 + grad(lifting)**2, mesh))
    lifting_norms.append(value)
    print(f"{name}: full H1 norm = {value:.6f}")

Draw(u_g_min, mesh, "minimum full-H1-norm lifting");

## 2. Dirichlet, Neumann, and Robin conditions together

Use the exact solution

$$
\begin{align*}
u_{\rm ex}=x(1+y),\qquad -\Delta u_{\rm ex}=0.
\end{align*}
$$

The boundary conditions are

| Boundary | Type | Data |
|---|---|---|
| left | Dirichlet | $u=0$ |
| top | Neumann | $\partial_n u=x$ |
| bottom | Neumann | $\partial_n u=-x$ |
| right | Robin | $\partial_n u+\eta u=(1+\eta)(1+y)$ |

We start with $\eta=1$. Since $\nabla u_{\rm ex}=(1+y,x)$, the outward normals explain the opposite Neumann signs at the top and bottom. The Robin datum changes with $\eta$ if we want to keep this same exact solution.

The test functions vanish only on the left edge. Integration by parts gives

$$
\begin{align*}
\mathcal A(u,v)
&=\int_\Omega\nabla u\cdot\nabla v\,dx
  +\int_{\Gamma_R}\eta uv\,ds,\\
\ell(v)
&=\int_{\Gamma_{\rm top}}xv\,ds
  -\int_{\Gamma_{\rm bottom}}xv\,ds
  +\int_{\Gamma_R}(1+\eta)(1+y)v\,ds.
\end{align*}
$$

Neumann and Robin data are assembled in the load. Only the Dirichlet edge is passed to `H1(..., dirichlet=...)`. Thus, they are called **essential** boundary conditions, whereas the others are called **natural**, as they are fulfilled only in a weak sense.

In [ ]:
eta = 1.0
u_M_exact = CF(x*(1+y))
h_R = (1+eta)*(1+y)

V_M = H1(mesh, order=order, dirichlet="left")
u_M, v_M = V_M.TnT()

a_M = BilinearForm(V_M, symmetric=True)
a_M += grad(u_M)*grad(v_M)*dx + eta*u_M*v_M*ds("right")
l_M = LinearForm(V_M)
l_M += x*v_M*ds("top") - x*v_M*ds("bottom") + h_R*v_M*ds("right")

a_M.Assemble()
l_M.Assemble()
uh_M = GridFunction(V_M, name="mixed boundary solution")
uh_M.vec.data = a_M.mat.Inverse(V_M.FreeDofs(), inverse="sparsecholesky")*l_M.vec

l2_M, h1_M = error_norms(uh_M, u_M_exact)
print(f"L2 error: {l2_M:.3e}; full H1 error: {h1_M:.3e}")
Draw(uh_M, mesh, "mixed Dirichlet-Neumann-Robin solution");

The exact solution has total degree two. On this straight-sided square, quadratic elements can represent it exactly. With `order = 2`, errors near rounding precision therefore check the signs and boundary regions in the weak formulation. For linear elements, a nonzero approximation error is expected even when the linear system is solved accurately.

## 3. The pure Neumann problem

For the pure Neumann Laplacian, the constant functions belong to the finite element space and have zero gradient. The stiffness matrix therefore has a kernel

$$
\begin{align*}
A\mathbf c=0,
\end{align*}
$$

where $\mathbf c$ is the coefficient vector of the **constant finite element function** $1$. We construct that function with `Set(1)`. For higher-order bases, its coefficient vector need not consist entirely of ones.

All degrees of freedom are now free, but this does not make the Laplace matrix invertible.

In [ ]:
V_N = H1(mesh, order=1)
u_N_kernel, v_N_kernel = V_N.TnT()

a_kernel = BilinearForm(V_N, symmetric=True)
a_kernel += grad(u_N_kernel)*grad(v_N_kernel)*dx
a_kernel.Assemble()

constant_N = GridFunction(V_N)
constant_N.Set(1)
print("vector constant_N.vec:", constant_N.vec)

kernel_vector = constant_N.vec.CreateVector()
kernel_vector.data = a_kernel.mat*constant_N.vec
kernel_residual_N = Norm(kernel_vector)
print(f"Norm of stiffness matrix times the constant mode: {kernel_residual_N:.3e}")

### Compatible data and a zero-mean exact solution

Choose

$$
\begin{align*}
u_{\rm ex}=x^2+y^2-\frac23,\qquad f=-4.
\end{align*}
$$

Then $\int_\Omega u_{\rm ex}\,dx=0$. Its normal derivative is $2$ on the right and top edges and $0$ on the left and bottom edges, which defines $h$.

The total load is

$$
\begin{align*}
\ell(v)=\int_\Omega f v\,dx+\int_{\partial\Omega}h v\,ds.
\end{align*}
$$

Testing with $v=1$ gives the compatibility condition

$$
\begin{align*}
\ell(1)=\int_\Omega f\,dx+\int_{\partial\Omega}h\,ds=-4+4=0.
\end{align*}
$$

In [ ]:
u_N_exact = CF(x*x + y*y - 2/3)
f_N = CF(-4)

source_N = Integrate(f_N, mesh)
flux_N = Integrate(2, mesh, definedon=mesh.Boundaries("right|top"))
compatibility_N = source_N + flux_N

print(f"Volume load: {source_N:.6f}; boundary load: {flux_N:.6f}")
print(f"Compatibility residual ell(1): {compatibility_N:.3e}")

### Removing the constant ambiguity with a mean constraint

We impose $\int_\Omega u_h\,dx=0$ using one scalar Lagrange multiplier $\lambda_h$. `NumberSpace(mesh)` supplies this single global unknown. The product space is $V_h\times\mathbb R$ and the augmented problem is

$$
\begin{align*}
\int_\Omega\nabla u_h\cdot\nabla v_h\,dx
+\lambda_h\int_\Omega v_h\,dx
+\mu\int_\Omega u_h\,dx=\ell(v_h)
\quad\forall(v_h,\mu)\in V_h\times\mathbb R.
\end{align*}
$$

In matrix form, with $m_i=\int_\Omega\varphi_i\,dx$,

$$
\begin{align*}
\begin{pmatrix}A&m\\m^\top&0\end{pmatrix}
\binom{\mathbf u}{\lambda_h}
=\binom{\mathbf b}{0}.
\end{align*}
$$

This matrix is symmetric but **indefinite**. We use the general direct solver `umfpack`. With compatible data the multiplier is zero, and the first component is the unique solution with mean zero.

In [ ]:
X_N = V_N * NumberSpace(mesh)
(u_N, lam_N), (v_N, mu_N) = X_N.TnT()

a_N = BilinearForm(X_N, symmetric=True)
a_N += (grad(u_N)*grad(v_N) + lam_N*v_N + mu_N*u_N)*dx
l_N = LinearForm(X_N)
l_N += f_N*v_N*dx + 2*v_N*ds("right|top")

a_N.Assemble()
l_N.Assemble()
inverse_N = a_N.mat.Inverse(X_N.FreeDofs(), inverse="umfpack")
gf_N = GridFunction(X_N)
gf_N.vec.data = inverse_N*l_N.vec
uh_N, lambda_N = gf_N.components

mean_N = Integrate(uh_N, mesh)/Integrate(1, mesh)
multiplier_N = lambda_N.vec[0]
l2_N, h1_N = error_norms(uh_N, u_N_exact)
print(f"Mean of u_h: {mean_N:.3e}; multiplier: {multiplier_N:.3e}")
print(f"L2 error: {l2_N:.3e}; full H1 error: {h1_N:.3e}")
Draw(uh_N, mesh, "Neumann solution with mean zero");
Draw(uh_N+1, mesh, "another Neumann solution: u_h + 1");

### What happens for incompatible data?

Now replace $f=-4$ by $f=-3$, keeping the boundary data unchanged. Then $\ell(1)=1$, so the original Neumann problem has **no solution**.

The augmented matrix is still invertible. Testing its first equation with $v_h=1$ shows

$$
\begin{align*}
\lambda_h|\Omega|=\ell(1).
\end{align*}
$$

A nonzero multiplier therefore reveals that the augmented problem has corrected the source to $f-\lambda_h$. On our unit square, the new multiplier is $1$ and the corrected source is again $-4$. We should obtain the same mean-zero first component as before, but it is **not** a solution of the incompatible problem with source $-3$.

In [ ]:
(u_N_bad, lam_N_bad), (v_N_bad, mu_N_bad) = X_N.TnT()
l_N_bad = LinearForm(X_N)
l_N_bad += (f_N+1)*v_N_bad*dx + 2*v_N_bad*ds("right|top")
l_N_bad.Assemble()

gf_N_bad = GridFunction(X_N)
gf_N_bad.vec.data = inverse_N*l_N_bad.vec
uh_N_bad, lambda_N_bad = gf_N_bad.components
compatibility_bad = Integrate(f_N+1, mesh) + flux_N
multiplier_bad = float(lambda_N_bad.vec[0])
unchanged_solution_bad = sqrt(Integrate((uh_N_bad-uh_N)**2, mesh))

print(f"Compatibility residual: {compatibility_bad:.6f}")
print(f"Multiplier: {multiplier_bad:.6f}; expected ell(1)/area: {compatibility_bad/Integrate(1, mesh):.6f}")
print(f"L2 difference of the corrected first components: {unchanged_solution_bad:.3e}")
print("The original Neumann data are incompatible, but the multiplier corrects the load.")

## 4. When convection matters: stability and boundary layers

The smallness assumption on $\vec b$ in Chapter 4 is **sufficient**, not necessary. A large convection field does not automatically make the continuous PDE or its Galerkin discretization singular.

We separate two effects:

- On a very coarse mesh, a field with positive divergence can make the **discrete Galerkin problem** singular. A divergence-free field of the same pointwise magnitude avoids this loss of coercivity.
- A large constant, divergence-free field can still change the solution dramatically.

### 4a. A controlled discrete breakdown

First solve

$$
\begin{align*}
-\Delta u+\vec b\cdot\nabla u+u=1\quad\text{in }\Omega,
\qquad u=0\quad\text{on }\partial\Omega.
\end{align*}
$$

We compare radial flow and rotation about the centre:

$$
\begin{align*}
\vec b_{\rm rad}
&=\kappa\binom{x-1/2}{y-1/2},
&\operatorname{div}\vec b_{\rm rad}&=2\kappa,\\
\vec b_{\rm rot}
&=\kappa\binom{-(y-1/2)}{x-1/2},
&\operatorname{div}\vec b_{\rm rot}&=0.
\end{align*}
$$

The two fields have the **same length at every point**. For a homogeneous Dirichlet test function,

$$
\begin{align*}
\int_\Omega(\vec b\cdot\nabla v)v\,dx
=-\frac12\int_\Omega(\operatorname{div}\vec b)v^2\,dx.
\end{align*}
$$

Consequently,

$$
\begin{align*}
\mathcal A_{\rm rad}(v,v)
&=\|\nabla v\|_{L^2}^2+(1-\kappa)\|v\|_{L^2}^2,\\
\mathcal A_{\rm rot}(v,v)
&=\|\nabla v\|_{L^2}^2+\|v\|_{L^2}^2.
\end{align*}
$$

The divergence-free form remains coercive for every $\kappa$. Positive divergence can remove the coercivity guarantee. Full Dirichlet data are important here: they make the boundary term from the convection identity vanish.

In [ ]:
arrow_grid = np.linspace(0.05, 0.95, 9)
XX, YY = np.meshgrid(arrow_grid, arrow_grid)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, title, bx, by in [
    (axes[0], "Radial field: div b = 2 kappa", XX-0.5, YY-0.5),
    (axes[1], "Rotation: div b = 0", -(YY-0.5), XX-0.5),
]:
    ax.quiver(XX, YY, bx, by, angles="xy", scale_units="xy", scale=3)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="x", ylabel="y", title=title)
    ax.set_aspect("equal")
fig.tight_layout()
plt.show()

Use a square split into **four triangles meeting at the centre**, and linear elements with zero values on the outer boundary. There is exactly one free degree of freedom. Its hat function $\varphi$ is one at the centre and zero at the four corners, so every discrete solution has the form $u_h=U\varphi$.

Direct integration gives

$$
\begin{align*}
\int_\Omega |\nabla\varphi|^2\,dx=4,
\qquad \int_\Omega\varphi^2\,dx=\frac16,
\qquad \ell(\varphi)=\int_\Omega\varphi\,dx=\frac13.
\end{align*}
$$

Thus the free matrix is a single number

$$
\begin{align*}
A_{\rm rad}(\kappa)=\frac{25-\kappa}{6},
\qquad A_{\rm rot}(\kappa)=\frac{25}{6}.
\end{align*}
$$

We will assemble and solve the finite element systems rather than inserting these formulas into the solver.

In [ ]:
from netgen.meshing import Mesh as NetgenMesh, MeshPoint, Pnt, Element2D, Element1D

ngmesh_star = NetgenMesh(dim=2)
star_coordinates = np.array([(0, 0), (1, 0), (1, 1), (0, 1), (0.5, 0.5)])
star_points = [ngmesh_star.Add(MeshPoint(Pnt(px, py, 0)))
               for px, py in star_coordinates]
star_domain = ngmesh_star.AddRegion("domain", dim=2)
star_outer = ngmesh_star.AddRegion("outer", dim=1)
star_triangles = np.array([(i, (i+1) % 4, 4) for i in range(4)])
for i in range(4):
    ngmesh_star.Add(Element2D(star_domain,
        [star_points[i], star_points[(i+1) % 4], star_points[4]]))
    ngmesh_star.Add(Element1D(
        [star_points[i], star_points[(i+1) % 4]], index=star_outer))
ngmesh_star.Compress()
mesh_star = Mesh(ngmesh_star)

# This experiment intentionally uses P1, independently of the global order.
V_star = H1(mesh_star, order=1, dirichlet="outer")
u_star, v_star = V_star.TnT()
free_star = [i for i, is_free in enumerate(V_star.FreeDofs()) if is_free]
centre_dof = free_star[0]
phi_star = GridFunction(V_star)
phi_star.vec[centre_dof] = 1

l_star = LinearForm(V_star)
l_star += v_star*dx
l_star.Assemble()
print(f"Triangles: {mesh_star.ne}; free degrees of freedom: {len(free_star)}")
print("Stiffness, mass, load:",
      Integrate(grad(phi_star)**2, mesh_star),
      Integrate(phi_star**2, mesh_star), l_star.vec[centre_dof])
Draw(mesh_star);

For the radial field, the predicted amplitude is $U=2/(25-\kappa)$: it grows near $\kappa=25$, the discrete equation is inconsistent at $25$, and the amplitude becomes negative above $25$. For rotation, this particular one-dimensional discrete space gives $U=2/25$ for all $\kappa$.

At the singular value we inspect the zero free matrix and its nonzero load, and **do not attempt to factor it**. 

This is a failure of the chosen coarse Galerkin approximation.

In [ ]:
star_directions = {
    "radial": CF((x-0.5, y-0.5)),
    "rotation": CF((-(y-0.5), x-0.5)),
}
kappa_values = [0.0, 20.0, 24.0, 24.9, 25.0, 26.0]
star_diagonals = {name: [] for name in star_directions}
star_amplitudes = {name: [] for name in star_directions}
star_solutions = {}
star_singular_cases = []

for kappa in kappa_values:
    labels = []
    for name, direction in star_directions.items():
        a_star = BilinearForm(V_star, symmetric=False)
        a_star += (grad(u_star)*grad(v_star) + kappa*(direction*grad(u_star))*v_star + u_star*v_star)*dx
        a_star.Assemble()
        diagonal = a_star.mat[centre_dof, centre_dof]
        star_diagonals[name].append(diagonal)
        if np.isclose(diagonal, 0, atol=1e-12):
            star_amplitudes[name].append(np.nan)
            star_singular_cases.append((name, kappa))
            labels.append(f"{name}: singular, nonzero load")
            continue
        uh_star = GridFunction(V_star)
        uh_star.vec.data = a_star.mat.Inverse(
            V_star.FreeDofs(), inverse="umfpack")*l_star.vec
        amplitude = uh_star(mesh_star(0.5, 0.5))
        star_amplitudes[name].append(amplitude)
        star_solutions[(name, kappa)] = uh_star
        labels.append(f"{name}: centre value = {amplitude:.4g}")
    print(f"kappa = {kappa:g}: " + "; ".join(labels))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for interval in [np.linspace(0, 24.9, 300), np.linspace(25.1, 30, 200)]:
    ax.plot(interval, 2/(25-interval), color="tab:red", linewidth=1)
ax.scatter(kappa_values, star_amplitudes["radial"], color="tab:red",
           label="radial: assembled FE solutions", zorder=3)
ax.plot(kappa_values, star_amplitudes["rotation"], "o-", color="tab:blue",
        label="rotation: assembled FE solutions")
ax.axvline(25, color="black", linestyle="--", label="radial system singular")
ax.set(xlabel="kappa", ylabel="u_h at the centre", xlim=(0, 30), ylim=(-22, 22),
       title="Same field magnitude, different discrete stability")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

### 4b. Large divergence-free convection creates an outflow layer

Now keep the **source and boundary data fixed** and solve

$$
\begin{align*}
-\Delta u+B\,\partial_xu&=0&&\text{in }(0,1)^2,\\
u&=0&&\text{on the left},\\
u&=1&&\text{on the right},\\
\partial_nu&=0&&\text{on the top and bottom}.
\end{align*}
$$

The field $\vec b=(B,0)$ is divergence-free for every $B$. The test functions vanish at the left and right, while $\vec b\cdot n=0$ on the other two edges. Therefore

$$
\begin{align*}
\int_\Omega B\,\partial_xv\,v\,dx=0,
\qquad \mathcal A(v,v)=\|\nabla v\|_{L^2(\Omega)}^2.
\end{align*}
$$

The form remains coercive on the homogeneous test space, regardless of the size of $B$.

For $B>0$, the exact solution is

$$
\begin{align*}
u_B(x,y)=\frac{e^{Bx}-1}{e^B-1}.
\end{align*}
$$

For $B=0$, it is $u_0=x$. With increasing $B$, the interior value approaches zero and the transition to the prescribed value $1$ takes place in an outflow layer of thickness about $1/B$.

In [ ]:
from ngsolve.meshes import MakeStructured2DMesh

# Resolve the layer in x (the exact solution is independent of y).
nx_layer, ny_layer = 30, 8
mesh_layer = MakeStructured2DMesh(nx=nx_layer, ny=ny_layer, quads=False)
V_layer = H1(mesh_layer, order=1, dirichlet="left|right")
u_layer, v_layer = V_layer.TnT()
Draw(mesh_layer);

We use the lifting correction from Part 1. The coefficient function $g=x$ supplies the values $0$ and $1$ on the two Dirichlet edges. The homogeneous Neumann conditions on the top and bottom are already encoded by the weak form.

Only $B$ changes between the solves. No forcing or boundary datum is recomputed to preserve an exact solution. For numerical evaluation of the exact exponential profile, an equivalent expression with nonpositive exponents avoids overflow.

In [ ]:
B_layer_values = [0.0, 5.0, 30.0]
layer_profile_x = np.linspace(0, 1, 401)
layer_solutions = {}
layer_profiles = {}
layer_exact_profiles = {}
layer_l2_errors = {}

for B_value in B_layer_values:
    a_layer = BilinearForm(V_layer, symmetric=False)
    a_layer += (grad(u_layer)*grad(v_layer)
                + B_value*grad(u_layer)[0]*v_layer)*dx
    a_layer.Assemble()
    uh_layer = GridFunction(V_layer)
    uh_layer.Set(x, BND)
    rhs_layer = uh_layer.vec.CreateVector()
    rhs_layer.data = -a_layer.mat*uh_layer.vec
    uh_layer.vec.data += a_layer.mat.Inverse(
        V_layer.FreeDofs(), inverse="umfpack")*rhs_layer
    layer_solutions[B_value] = uh_layer
    layer_profiles[B_value] = np.array([
        uh_layer(mesh_layer(float(xi), 0.5)) for xi in layer_profile_x])
    if B_value == 0:
        exact_layer = CF(x)
        layer_exact_profiles[B_value] = layer_profile_x
    else:
        exact_layer = (exp(B_value*(x-1))-exp(-B_value))/(1-exp(-B_value))
        layer_exact_profiles[B_value] = (
            np.exp(B_value*(layer_profile_x-1))-np.exp(-B_value))/(1-np.exp(-B_value))
    layer_l2_errors[B_value] = sqrt(Integrate(
        (uh_layer-exact_layer)**2, mesh_layer, order=10))
    print(f"B = {B_value:g}: u_h(0.9, 0.5) = {uh_layer(mesh_layer(0.9, 0.5)):.4f}")
    Draw(uh_layer, mesh_layer, f"Outflow layer, B = {B_value:g}");

In [ ]:
fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(11, 4))
for B_value in B_layer_values:
    line, = ax_full.plot(layer_profile_x, layer_profiles[B_value], label=f"B = {B_value:g}")
    colour = line.get_color()
    ax_full.plot(layer_profile_x, layer_exact_profiles[B_value], "--",
                 color=colour, linewidth=1)
    ax_zoom.plot(layer_profile_x, layer_profiles[B_value], color=colour)
    ax_zoom.plot(layer_profile_x, layer_exact_profiles[B_value], "--",
                 color=colour, linewidth=1)
ax_full.set(xlim=(0, 1), ylim=(-0.2, 1.02), title="Whole interval",
            xlabel="x", ylabel="u(x, 0.5)")
ax_zoom.set(xlim=(0.8, 1), ylim=(-0.05, 1.02), title="Zoom near the outflow boundary",
            xlabel="x", ylabel="u(x, 0.5)")
for ax in (ax_full, ax_zoom):
    ax.grid(True, alpha=0.3)
ax_full.legend()
fig.suptitle("Large divergence-free convection: solid = FE, dashed = exact")
fig.tight_layout()
plt.show()


[Lecture notebooks](../README.md) · [Lecture notes](../../script/README.md) · [Course overview](../../index.ipynb)